In [1]:
from datasets import load_dataset

# ==========================
#    DATASET FOR TESTING
# ==========================

SAMPLE_SIZE = 3000000
OUTPUT_SAMPLE_PATH = "WLOSKI_DATASET_SAMPLE.txt"

print(f"Downloading sample ({SAMPLE_SIZE} articles)...")

dataset_sample = load_dataset(
    "wikimedia/wikipedia", "20231101.it", split=f"train[:{SAMPLE_SIZE}]"
)

with open(OUTPUT_SAMPLE_PATH, "w", encoding="utf-8") as f:
  for item in dataset_sample:
    f.write(item["text"] + "\n\n")

print(f"Done! Saved {len(dataset_sample)} articles to {OUTPUT_SAMPLE_PATH}")
print(f"Preview of the first article:\n{dataset_sample[0]['text'][:300]}...")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

C:\Users\jakub\miniconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jakub\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\jakub\miniconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfa

20231101.it/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  529MB            

20231101.it/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

20231101.it/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  376MB            

...

20231101.it/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

20231101.it/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

20231101.it/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1833639 [00:00<?, ? examples/s]

Done! Saved 1833639 articles to WLOSKI_DATASET_SAMPLE.txt
Preview of the first article:
Larmonium, o harmonium, è un tipo di organo a serbatoio d'aria con una (o più) tastiere e due pedali per azionare i mantici per l'aria. Il suono è prodotto da una o più file di ance libere, analogamente alla fisarmonica e all'armonica a bocca. 

Nella classificazione Hornbostel-Sachs, rientra nella ...


In [ ]:
from datasets import load_dataset

# ==========================
#   FULL DATASET (~5GB)
# ==========================

print(f"Downloading full dataset!")

dataset_full = load_dataset("wikimedia/wikipedia", "20231101.it", split="train")

print(
    f"Installed, amount of articles:"
    f" {len(dataset_full):,}"
)

In [ ]:
import os
import re
import urllib.request

# ==========================
#       KONFIGURACJA
# ==========================
INPUT_SAMPLE_PATH = "WLOSKI_DATASET_SAMPLE.txt"
OUTPUT_WORDS_PATH = "WLOSKI_SLOWA_OCZYSZCZONE.txt"
DICT_FILE = "it_IT.dic"

# 1. Pobranie słownika języka włoskiego, jeśli go brakuje
if not os.path.exists(DICT_FILE):
    print("Pobieranie oficjalnego słownika języka włoskiego...")
    url = "https://raw.githubusercontent.com/LibreOffice/dictionaries/master/it_IT/it_IT.dic"
    urllib.request.urlretrieve(url, DICT_FILE)
    print("Słownik pobrany.")

print("Wczytywanie słownika do pamięci RAM...")
italian_lexicon = set()
with open(DICT_FILE, "r", encoding="latin-1") as f:
    for line in f:
        # Pliki Hunspell mają format 'słowo/flagi' — bierzemy samo słowo
        word = line.strip().split("/")[0].lower()
        if word:
            italian_lexicon.add(word)

print(f"Załadowano {len(italian_lexicon):,} bazowych włoskich słów.")

# 2. Wyrażenia regularne do czyszczenia
HYPHENS_AND_APOSTROPHES = re.compile(r"[-–—'’]")
NON_LETTER_CLEANER = re.compile(r"[^a-zàèéìòù\s]")
ITALIAN_WORD_REGEX = re.compile(r"^[a-zàèéìòù]+$")

# Włoskie spójniki, przyimki i formy elizyjne (zaimki/rodzajniki po odcięciu apostrofu)
ALLOWED_SHORT_WORDS = {
    "a", "e", "i", "o", "u",
    "d", "l", "c", "m", "t", "s", "v", "n"
}

# Inicjalizacja zbiorów
valid_words = set(ALLOWED_SHORT_WORDS)
rejected_words = set()

print(f"\nPrzetwarzanie pliku {INPUT_SAMPLE_PATH}...")

line_count = 0
with open(INPUT_SAMPLE_PATH, "r", encoding="utf-8") as f_in:
    for line in f_in:
        line_count += 1
        if line_count % 500000 == 0:
            print(f"Przetworzono {line_count:,} linii... Znaleziono {len(valid_words):,} poprawnych słów.")

        text = line.lower()

        # Rozbicie myślników i apostrofów na spacje (np. socio-economico -> socio economico, l'arte -> l arte)
        text = HYPHENS_AND_APOSTROPHES.sub(" ", text)

        # Usunięcie cyfr, znaków interpunkcyjnych i znaków specjalnych
        text = NON_LETTER_CLEANER.sub(" ", text)

        # Weryfikacja słów
        for token in text.split():
            if token in ALLOWED_SHORT_WORDS:
                continue

            if len(token) == 1:
                continue

            if not ITALIAN_WORD_REGEX.match(token):
                continue

            if token in valid_words or token in rejected_words:
                continue

            # Sprawdzenie w słowniku w czasie O(1)
            if token in italian_lexicon:
                valid_words.add(token)
            else:
                rejected_words.add(token)

print(f"\nKoniec! Zebrano łącznie {len(valid_words):,} unikalnych słów.")
print(f"Zapisywanie do {OUTPUT_WORDS_PATH}...")

with open(OUTPUT_WORDS_PATH, "w", encoding="utf-8") as f_out:
    for word in sorted(valid_words):
        f_out.write(word + "\n")

print("Zapis zakończony sukcesem.")

Pobieranie oficjalnego słownika języka włoskiego...
Słownik pobrany.
Wczytywanie słownika do pamięci RAM...
Załadowano 95,210 bazowych włoskich słów.

Przetwarzanie pliku WLOSKI_DATASET_SAMPLE.txt...
Przetworzono 500,000 linii... Znaleziono 39,687 poprawnych słów.
Przetworzono 1,000,000 linii... Znaleziono 46,411 poprawnych słów.
Przetworzono 1,500,000 linii... Znaleziono 49,601 poprawnych słów.
